In [1]:
with open(r"C:\Users\jeffe\OneDrive\Desktop\Portfolio Projects\archive\2018-04-12_batchdata_updated_struct_errorcorrect.mat", "rb") as f:
    print(f.read(16))

b'MATLAB 7.3 MAT-f'


In [2]:
file_path = r"C:\Users\jeffe\OneDrive\Desktop\Portfolio Projects\archive\2018-04-12_batchdata_updated_struct_errorcorrect.mat"

In [4]:
"""
extract_summary.py

WHAT THIS DOES (read this before running it):
The raw Severson battery dataset stores, for every cell, every single
cycle's full voltage/current/temperature curve. That's why the file is
huge (~3GB) - it's hundreds of cells x thousands of measurements each.

We don't need all of that to get started. We need ONE row per battery
cell, containing:
  - cycle_life: how many cycles the cell survived before failing (this is
    our TARGET - what we're trying to predict)
  - a handful of summary stats from the cell's early cycles (our FEATURES)

Running this script turns a multi-GB file into a tiny CSV (a few KB) that
is safe and fast to upload anywhere. This is exactly what a data
engineer does with any large raw dataset: extract only what's needed.

HOW TO RUN THIS:
1. Make sure you have Python installed (check by typing `python --version`
   in a terminal/command prompt). If not, the easiest path is Google
   Colab (colab.research.google.com) - no install needed, just upload
   your .mat file there instead and run this same code in a notebook cell.
2. Install one library: pip install h5py pandas
3. Edit the FILE_PATH variable below to point at your downloaded file.
4. Run: python extract_summary.py
5. Upload the resulting `battery_summary.csv` file to our chat.
"""

import h5py
import pandas as pd
import numpy as np

# EDIT THIS to match your actual downloaded file's path and name
FILE_PATH = "2017-05-12_batchdata_updated_struct_errorcorrect.mat"

def extract_battery_summaries(file_path):
    """
    Opens the MATLAB v7.3 (HDF5) file and pulls out, for each battery cell:
    - cycle_life (the target we want to predict)
    - a few summary features from early cycles (what we'll use to predict it)
    """
    rows = []

    with h5py.File(file_path, 'r') as f:
        batch = f['batch']
        num_cells = batch['summary'].shape[0]
        print(f"Found {num_cells} battery cells in this file.")

        for i in range(num_cells):
            try:
                # cycle_life: total cycles survived (our prediction target)
                cl_ref = f[batch['cycle_life'][i, 0]]
                cycle_life = int(np.array(cl_ref)[0][0])

                # summary struct: has arrays of stats, one entry per cycle.
                # NOTE: cycle_life needed to be "dereferenced" (followed as a
                # pointer), but QDischarge/IR do NOT - they're stored directly
                # inside the summary group. Mixed storage styles like this are
                # common in nested scientific data files - always verify each
                # level rather than assuming it matches the level above.
                summary_ref = f[batch['summary'][i, 0]]
                qdischarge = np.array(summary_ref['QDischarge']).flatten()
                ir = np.array(summary_ref['IR']).flatten()

                # Only use the first 100 cycles - this matches our business
                # goal: predict from EARLY data, not the full life of the cell
                early_cycles = min(100, len(qdischarge))

                row = {
                    "cell_id": i,
                    "cycle_life": cycle_life,
                    "capacity_at_cycle_10": qdischarge[9] if len(qdischarge) > 9 else np.nan,
                    "capacity_at_cycle_100": qdischarge[early_cycles - 1],
                    "capacity_fade_10_to_100": qdischarge[9] - qdischarge[early_cycles - 1] if len(qdischarge) > 9 else np.nan,
                    "internal_resistance_at_100": ir[early_cycles - 1] if len(ir) >= early_cycles else np.nan,
                }
                rows.append(row)
            except Exception as e:
                print(f"Skipped cell {i}, had an issue: {e}")

    return pd.DataFrame(rows)


if __name__ == "__main__":
    df = extract_battery_summaries(file_path)
    print(df.head())
    print(f"\nExtracted {len(df)} battery summaries.")
    df.to_csv("battery_summary.csv", index=False)
    print("Saved to battery_summary.csv - this is the file to upload to Claude.")

Found 46 battery cells in this file.
Skipped cell 23, had an issue: cannot convert float NaN to integer
Skipped cell 32, had an issue: cannot convert float NaN to integer
   cell_id  cycle_life  capacity_at_cycle_10  capacity_at_cycle_100  \
0        0        1009              1.070419               1.069454   
1        1        1063              1.068262               1.066478   
2        2        1267              1.062787               1.060614   
3        3        1115              1.065105               1.063595   
4        4        1048              1.074539               1.073859   

   capacity_fade_10_to_100  internal_resistance_at_100  
0                 0.000965                    0.015271  
1                 0.001784                    0.015059  
2                 0.002174                    0.014477  
3                 0.001511                    0.015063  
4                 0.000680                    0.015180  

Extracted 44 battery summaries.
Saved to battery_summary.cs

In [ ]:
import h5py

with h5py.File(file_path, 'r') as f:
    print("Top-level keys:", list(f.keys()))
    print()
    print("Fields inside 'batch':", list(f['batch'].keys()))
    print()
    cl = f['batch']['cycle_life']
    print("cycle_life shape/dtype:", cl.shape, cl.dtype)
    print("cycle_life[0,0]:", cl[0, 0], "-- type:", type(cl[0, 0]))
    print()
    summ = f['batch']['summary']
    print("summary shape/dtype:", summ.shape, summ.dtype)

In [ ]:
with h5py.File(file_path, 'r') as f:
    summary_ref = f[f['batch']['summary'][0, 0]]
    print("Fields in summary:", list(summary_ref.keys()))
    qd = np.array(summary_ref['QDischarge']).flatten()
    print("QDischarge shape:", qd.shape)
    print("First 5 values:", qd[:5])

In [ ]:
df = extract_battery_summaries(file_path)
print(df.head())
print(f"\nExtracted {len(df)} battery summaries.")
df.to_csv("battery_summary.csv", index=False)

In [5]:
import h5py

with h5py.File(file_path, 'r') as f:
    cycles_ref = f[f['batch']['cycles'][0, 0]]  # cell 0's cycles
    print("Fields inside cycles:", list(cycles_ref.keys()))
    print()
    
    # Check cycle 10 and cycle 100's voltage (V) and discharge capacity (Qd)
    v_field = cycles_ref['V']
    qd_field = cycles_ref['Qd']
    print("V field shape/dtype:", v_field.shape, v_field.dtype)
    print("Qd field shape/dtype:", qd_field.shape, qd_field.dtype)

Fields inside cycles: ['I', 'Qc', 'Qd', 'Qdlin', 'T', 'Tdlin', 'V', 'discharge_dQdV', 't']

V field shape/dtype: (1008, 1) object
Qd field shape/dtype: (1008, 1) object


In [6]:
with h5py.File(file_path, 'r') as f:
    cycles_ref = f[f['batch']['cycles'][0, 0]]
    dqdv_field = cycles_ref['discharge_dQdV']
    print("discharge_dQdV shape/dtype:", dqdv_field.shape, dqdv_field.dtype)
    
    # Follow the reference for cycle 10 (index 9) and look at the actual curve
    cycle10_dqdv = np.array(f[dqdv_field[9, 0]]).flatten()
    print("Cycle 10 dQdV curve length:", len(cycle10_dqdv))
    print("First 10 values:", cycle10_dqdv[:10])

discharge_dQdV shape/dtype: (1008, 1) object
Cycle 10 dQdV curve length: 1000
First 10 values: [-0.01225278 -0.01225278 -0.01225278 -0.01225278 -0.01225278 -0.01225278
 -0.01225278 -0.01225278 -0.01225278 -0.01225278]


In [7]:
with h5py.File(file_path, 'r') as f:
    cycles_ref = f[f['batch']['cycles'][0, 0]]
    dqdv_field = cycles_ref['discharge_dQdV']
    c10 = np.array(f[dqdv_field[9, 0]]).flatten()
    c100 = np.array(f[dqdv_field[99, 0]]).flatten()
    diff = c100 - c10
    print("Variance of difference curve:", np.var(diff))

Variance of difference curve: 0.0018571950612528377


In [9]:
with h5py.File(file_path, 'r') as f:
    cycles_ref = f[f['batch']['cycles'][0, 0]]
    qdlin_field = cycles_ref['Qdlin']
    print("Qdlin shape/dtype:", qdlin_field.shape, qdlin_field.dtype)
    c10 = np.array(f[qdlin_field[9, 0]]).flatten()
    c100 = np.array(f[qdlin_field[99, 0]]).flatten()
    print("Curve length:", len(c10))
    diff = c100 - c10
    print("Variance of Qdlin difference curve:", np.var(diff))

Qdlin shape/dtype: (1008, 1) object
Curve length: 1000
Variance of Qdlin difference curve: 5.687024898905241e-05
